In [ ]:
code = 'SRE_PREMIUM_SHIFT'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_TILL_PRM_SeparateLegSL_&_ExitAtSLPrice_output\\'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def SRE_SHIFT(bt, start_time, end_time, orderside, method, om, divider, movement):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        end_dt_1m = end_dt + datetime.timedelta(minutes=10)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        entry_time = start_dt
        premium = ce_price + pe_price
        ce_start_dt, pe_start_dt = start_dt, start_dt
        
        from_candle_close = True if method == 'CC' else False
        
        trades = []
        shifting_pnl = 0
        exit_time = end_dt
        check_leg_sl = False
        for re in range(max_re+1):
        
            divider_price = cal_percent(premium, divider)
            move_price = cal_percent(premium, movement)
            
            start_dt = max(ce_start_dt, pe_start_dt)
            t_ce, t_pe = bt.get_straddle_data_no_backup(start_dt, end_dt, ce_scrip, pe_scrip, seperate=True)
            ce_data, pe_data = t_ce.copy(), t_pe.copy()

            _, _, _, _, ce_sl_time, _ = bt.sl_check_by_given_data(ce_data, sl_price=move_price, orderside=orderside, from_candle_close=from_candle_close)
            _, _, _, _, pe_sl_time, _ = bt.sl_check_by_given_data(pe_data, sl_price=move_price, orderside=orderside, from_candle_close=from_candle_close)

            ce_sl_time = ce_sl_time if ce_sl_time else end_dt_1m
            pe_sl_time = pe_sl_time if pe_sl_time else end_dt_1m

            if ce_sl_time < pe_sl_time:
                
                ce_price_at_sl = bt.options_data.loc[(ce_sl_time, ce_scrip), 'close']
                pe_price_at_sl = bt.options_data.loc[(ce_sl_time, pe_scrip), 'close']
                
                target_price = pe_price_at_sl + divider_price
                tpe_scrip, tpe_price, _, tpe_start_dt = bt.get_strike(ce_sl_time, end_dt, target=target_price, obove_target_only=True, only='PE')
                
                if tpe_scrip is None:
                    check_leg_sl = True
                    break

                tpremium = premium - pe_price_at_sl + tpe_price
                
                if int(tpe_scrip[:-2]) - int(ce_scrip[:-2]) > tpremium:
                    check_leg_sl = True
                    break
                else:
                    premium = tpremium
                    pnl = round((pe_price - pe_price_at_sl) - bt.Cal_slipage(pe_price), 2)
                    shifting_pnl += pnl
                    trades.extend([ce_scrip, ce_price, pe_scrip, pe_price, premium, divider_price, 'PE', ce_sl_time, pe_price_at_sl, ce_price_at_sl, pnl])
                    pe_scrip, pe_price, pe_start_dt = tpe_scrip, tpe_price, tpe_start_dt
                    

            elif pe_sl_time < ce_sl_time:
                
                ce_price_at_sl = bt.options_data.loc[(pe_sl_time, ce_scrip), 'close']
                pe_price_at_sl = bt.options_data.loc[(pe_sl_time, pe_scrip), 'close']
                
                target_price = ce_price_at_sl + divider_price
                tce_scrip, tce_price, _, tce_start_dt = bt.get_strike(pe_sl_time, end_dt, target=target_price, obove_target_only=True, only='CE')
                
                if tce_scrip is None:
                    check_leg_sl = True
                    break

                tpremium = premium - ce_price_at_sl + tce_price
                
                if int(pe_scrip[:-2]) - int(tce_scrip[:-2]) > tpremium:
                    check_leg_sl = True
                    break
                else:
                    premium = tpremium
                    pnl = round((ce_price - ce_price_at_sl) - bt.Cal_slipage(ce_price), 2)
                    shifting_pnl += pnl
                    trades.extend([ce_scrip, ce_price, pe_scrip, pe_price, premium, divider_price, 'CE', pe_sl_time, ce_price_at_sl, pe_price_at_sl, pnl])
                    ce_scrip, ce_price, ce_start_dt = tce_scrip, tce_price, tce_start_dt

            else:
                if ce_sl_time != end_dt_1m:
                    
                    ce_price_at_sl = bt.options_data.loc[(ce_sl_time, ce_scrip), 'close']
                    pe_price_at_sl = bt.options_data.loc[(ce_sl_time, pe_scrip), 'close']
                    
                    ce_pnl = round((ce_price - ce_price_at_sl) - bt.Cal_slipage(ce_price), 2)
                    shifting_pnl += ce_pnl
                    
                    pe_pnl = round((pe_price - pe_price_at_sl) - bt.Cal_slipage(pe_price), 2)
                    shifting_pnl += pe_pnl
                    
                    trades.extend([ce_scrip, ce_price, pe_scrip, pe_price, premium, divider_price, 'BOTH', ce_sl_time, ce_price_at_sl, pe_price_at_sl, ce_pnl+pe_pnl])
                    
                    target_price = pe_price_at_sl + divider_price
                    pe_scrip, pe_price, _, pe_start_dt = bt.get_strike(ce_sl_time, end_dt, target=target_price, obove_target_only=True, only='PE')
                    if pe_scrip is None: 
                        ce_scrip, exit_time = None, None
                        break
                        
                    target_price = ce_price_at_sl + divider_price
                    ce_scrip, ce_price, _, ce_start_dt = bt.get_strike(ce_sl_time, end_dt, target=target_price, obove_target_only=True, only='CE')
                    if ce_scrip is None:
                        pe_scrip, exit_time = None, None
                        break

                    premium = premium - ce_price_at_sl + ce_price - pe_price_at_sl + pe_price

                    if int(pe_scrip[:-2]) - int(ce_scrip[:-2]) > premium:
                        ce_scrip, pe_scrip, exit_time = None, None, None
                        ce_price, pe_price = '', ''
                        break
                    
                else:
                    re -= 1
                    exit_time = end_dt
                    break
                    
        if check_leg_sl:
            re -= 1
            if ce_scrip is not None:
                
                if ce_sl_time == end_dt_1m:
                    ce_sl_time = end_dt
                    eod_ce_price = bt.options.loc[(bt.options['scrip'] == ce_scrip) & (bt.options['date_time'] <= ce_sl_time), 'close'].iloc[-1]
                    eod_ce_pnl = round(ce_price - eod_ce_price - bt.Cal_slipage(ce_price), 2)
                else:
                    eod_ce_pnl = round(ce_price - move_price - bt.Cal_slipage(ce_price), 2)
            else:
                eod_ce_pnl = 0
                premium, divider_price = '', ''

            if pe_scrip is not None:
                
                if pe_sl_time == end_dt_1m:            
                    pe_sl_time = end_dt
                    eod_pe_price = bt.options.loc[(bt.options['scrip'] == pe_scrip) & (bt.options['date_time'] <= pe_sl_time), 'close'].iloc[-1]
                    eod_pe_pnl = round(pe_price - eod_pe_price - bt.Cal_slipage(pe_price), 2)
                else:
                    eod_pe_pnl = round(pe_price - move_price - bt.Cal_slipage(pe_price), 2)
            else:
                eod_pe_pnl = 0
                premium, divider_price = '', ''
        else:
            if ce_scrip is not None:

                if (exit_time == end_dt) or (exit_time == end_dt_1m):
                    eod_ce_price = bt.options.loc[(bt.options['scrip'] == ce_scrip) & (bt.options['date_time'] <= exit_time), 'close'].iloc[-1]
                    eod_ce_pnl = round(ce_price - eod_ce_price - bt.Cal_slipage(ce_price), 2)
                else:
                    eod_ce_pnl = round(ce_price - move_price - bt.Cal_slipage(ce_price), 2)
            else:
                eod_ce_pnl = 0
                premium, divider_price = '', ''

            if pe_scrip is not None:

                if (exit_time == end_dt) or (exit_time == end_dt_1m):
                    eod_pe_price = bt.options.loc[(bt.options['scrip'] == pe_scrip) & (bt.options['date_time'] <= exit_time), 'close'].iloc[-1]
                    eod_pe_pnl = round(pe_price - eod_pe_price - bt.Cal_slipage(pe_price), 2)
                else:
                    eod_pe_pnl = round(pe_price - move_price - bt.Cal_slipage(pe_price), 2)
            else:
                eod_pe_pnl = 0
                premium, divider_price = '', ''
            
        total_pnl = shifting_pnl + eod_ce_pnl + eod_pe_pnl
        trades.extend([ce_scrip, ce_price, pe_scrip, pe_price, premium, divider_price, '', '', '', '', eod_ce_pnl + eod_pe_pnl])
        
        for _ in range(max_re - re -1):
            trades.extend(['', '', '', '', '', '', '', '', '', '', 0])
            
        trades.extend([shifting_pnl, eod_ce_pnl, eod_pe_pnl, total_pnl])
        return [code, bt.index, start_time, end_time, orderside, method, om, divider, movement, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price] + trades

    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, method, om, divider, movement])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            max_re, re_entries = 20, 20
            
            log_cols = 'P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_Method/P_OM/P_Divider/P_Movement/Date/Day/DTE/EntryTime/Future'
            
            for r in range(max_re+1):
                log_cols += f'/CE{r}.Scrip/CE{r}.Price/PE{r}.Scrip/PE{r}.Price/Premium{r}/Divider{r}/ShiftLeg{r}/ShiftLeg{r}.Time/ShiftLeg{r}.Price.AT.SL/NonShift{r}.Price.At.SL/Shift{r}.PNL'
            
            log_cols += f'/Shifted.PNL/EOD.CE.PNL/EOD.PE.PNL/Total.PNL'
            log_cols = log_cols.split('/')
            
            for current_date in date_lists:
                
                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):
                    
                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")
                    
                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)
                    
                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)
                        
                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [SRE_SHIFT(bt, row['entry_time'], row['exit_time'], row['orderside'], row['method'], row['om'], row['divider'], row['movement']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))